In [13]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications.mobilenet import preprocess_input

In [14]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 2

In [15]:
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    r"D:\Final Year\Sem 2\Major Project\Model Project\Dataset\train\images",
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    r"D:\Final Year\Sem 2\Major Project\Model Project\Dataset\valid\images",
    label_mode='categorical',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

Found 10000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [16]:
train_data = train_data.map(lambda x, y: (preprocess_input(x), y))
val_data = val_data.map(lambda x, y: (preprocess_input(x), y))

In [17]:
AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(buffer_size=AUTOTUNE)
val_data = val_data.prefetch(buffer_size=AUTOTUNE)

In [18]:
base_model = MobileNet(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

In [19]:
for layer in base_model.layers:
    layer.trainable = False

In [20]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(NUM_CLASSES, activation='softmax')(x)

student_model = Model(inputs=base_model.input, outputs=output)

In [21]:
student_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [22]:
history = student_model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
313/313 [==============================] - 266s 833ms/step - loss: 0.0823 - accuracy: 0.9681 - val_loss: 0.0364 - val_accuracy: 0.9860
Epoch 2/5
313/313 [==============================] - 230s 735ms/step - loss: 0.0129 - accuracy: 0.9966 - val_loss: 0.0337 - val_accuracy: 0.9870
Epoch 3/5
313/313 [==============================] - 222s 709ms/step - loss: 0.0059 - accuracy: 0.9983 - val_loss: 0.0276 - val_accuracy: 0.9900
Epoch 4/5
313/313 [==============================] - 219s 700ms/step - loss: 0.0039 - accuracy: 0.9988 - val_loss: 0.0333 - val_accuracy: 0.9885
Epoch 5/5
313/313 [==============================] - 225s 717ms/step - loss: 0.0029 - accuracy: 0.9992 - val_loss: 0.0423 - val_accuracy: 0.9875


In [23]:
for layer in base_model.layers[-30:]:
    layer.trainable = True

student_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = student_model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
313/313 [==============================] - 398s 1s/step - loss: 0.0120 - accuracy: 0.9961 - val_loss: 0.0326 - val_accuracy: 0.9885
Epoch 2/5
313/313 [==============================] - 396s 1s/step - loss: 0.0079 - accuracy: 0.9972 - val_loss: 0.0293 - val_accuracy: 0.9900
Epoch 3/5
313/313 [==============================] - 399s 1s/step - loss: 0.0044 - accuracy: 0.9990 - val_loss: 0.0312 - val_accuracy: 0.9905
Epoch 4/5
313/313 [==============================] - 398s 1s/step - loss: 0.0034 - accuracy: 0.9991 - val_loss: 0.0373 - val_accuracy: 0.9880
Epoch 5/5
313/313 [==============================] - 403s 1s/step - loss: 0.0033 - accuracy: 0.9994 - val_loss: 0.0329 - val_accuracy: 0.9900


In [24]:
student_model.save("student_model.h5")

c:\Users\Hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [27]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet import preprocess_input

img_path = r"D:\Final Year\Sem 2\Major Project\Model Project\Dataset\valid\images\Non-Face\iphone.jpg"

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

prediction = student_model.predict(img_array)

print("Prediction:", prediction)
print("Class:", "Face" if np.argmax(prediction) == 0 else "Non-Face")

1/1 [==============================] - 0s 53ms/step
Prediction: [[2.8290390e-07 9.9999976e-01]]
Class: Non-Face


In [28]:
import os
print(os.path.getsize("mobilenet_model.pth") / (1024*1024), "MB")

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'mobilenet_model.pth'